# MethylGPT: CpG Imputation

This notebook demonstrates how to use MethylGPT's masked language model (MLM) head to **recover missing CpG methylation values**.

MethylGPT is pre-trained with a masked prediction objective: during training, a fraction of CpG values are masked and the model learns to predict them from the surrounding context. This same mechanism can be used at inference time to impute missing or held-out methylation values.

**Requirements:**
- A pre-trained MethylGPT model checkpoint
- Processed methylation data in parquet format
- CpG probe ID list (type3)
- GPU recommended for faster inference

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q methylgpt[tutorials]
    !pip install -q gdown
    import torch
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    else:
        print("WARNING: No GPU. Go to Runtime > Change runtime type > GPU")

In [ ]:
import os
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from methylgpt import MethylGPTModel, MethylVocab, create_dataloader
from methylgpt.model.methyl_pretraining import random_mask_value
from scgpt.tokenizer import tokenize_and_pad_batch

warnings.filterwarnings("ignore", message=".*IProgress.*")
warnings.filterwarnings("ignore", message=".*flash_attn.*")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Configure Paths

In [ ]:
# === UPDATE THESE PATHS ===
MODEL_DIR = Path("pretrained_models/methylgpt-medium")
CPG_LIST_FILE = "data/probe_ids_type3.csv"
PARQUET_DIR = "data/processed_type3_parquet_shuffled"

# On Colab: download model, probe IDs, and sample data
if IN_COLAB:
    import gdown, subprocess
    os.makedirs("data", exist_ok=True)

    # 1. Download pretrained model (medium)
    if not list(MODEL_DIR.glob("*.pt")) if MODEL_DIR.exists() else True:
        os.makedirs(str(MODEL_DIR), exist_ok=True)
        print("Downloading methylgpt-medium model...")
        gdown.download_folder(
            "https://drive.google.com/drive/folders/14M4wdS83el9PAgh9TdfjSCeEcDPbz34f",
            output=str(MODEL_DIR), quiet=True
        )
        print(f"Model downloaded to {MODEL_DIR}")
    else:
        print(f"Model already exists in {MODEL_DIR}")

    # 2. Download probe_ids_type3.csv
    if not os.path.exists(CPG_LIST_FILE):
        print("Downloading probe_ids_type3.csv...")
        subprocess.run([
            "wget", "-q", "-O", CPG_LIST_FILE,
            "https://www.dropbox.com/scl/fi/2n6bx7j8v0aon0kwfsghp/probe_ids_type3.csv?rlkey=ly133xlce1xxjiku6tiski6qq&st=pig4e41h&dl=1"
        ], check=True)
        print(f"Probe IDs saved to {CPG_LIST_FILE}")
    else:
        print(f"Probe IDs already exist at {CPG_LIST_FILE}")

    # 3. Download sample parquet data
    if not os.path.exists(PARQUET_DIR):
        print("Downloading sample parquet data (~2 GB)...")
        subprocess.run([
            "wget", "-q", "--show-progress", "-O", "data/parquet_data.tar.gz",
            "https://www.dropbox.com/scl/fi/bbs6sxlkpbx11rhyvdfto/processed_type3_parquet_shuffled.tar.gz?rlkey=s73utmumq6xldmv3y6kh9bz75&st=8pslwy2a&dl=1"
        ], check=True)
        subprocess.run(["tar", "-xzf", "data/parquet_data.tar.gz", "-C", "data/"], check=True)
        os.remove("data/parquet_data.tar.gz")
        print(f"Parquet data extracted to {PARQUET_DIR}")
    else:
        print(f"Parquet data already exists at {PARQUET_DIR}")


## 2. Load Model

In [ ]:
with open(MODEL_DIR / "args.json", "r") as f:
    config = json.load(f)

model_files = list(MODEL_DIR.glob("*.pt"))
assert model_files, f"No .pt found in {MODEL_DIR}"

config["load_model"] = True
config["pretrained_file"] = str(model_files[0])
config["mask_ratio"] = 0.15  # 15% masking for imputation evaluation

vocab = MethylVocab(
    probe_id_dir=CPG_LIST_FILE, pad_token="<pad>",
    special_tokens=["<pad>", "<cls>", "<eoc>"], save_dir=None,
)

model = MethylGPTModel.from_pretrained(config, vocab)
model.eval()
model.to(device)
if device.type == "cuda":
    model.half()

print(f"Model loaded: {config['layer_size']}-dim, {config['nlayers']} layers")

## 3. Prepare Data & Mask CpGs

To evaluate imputation quality, we take samples with known methylation values, **mask 15% of them**, and ask the model to predict the masked values. We then compare the predictions against the ground truth.

In [ ]:
# Load data
parquet_files = sorted([
    os.path.join(PARQUET_DIR, f) for f in os.listdir(PARQUET_DIR)
    if f.endswith(".parquet")
])
data_loader = create_dataloader([parquet_files[0]], batch_size=32)

# Run imputation evaluation
all_predicted = []
all_actual = []
all_masks = []

pad_value = config.get("pad_value", -2)
mask_value = config.get("mask_value", -1)
mask_ratio = config["mask_ratio"]

# Get model dtype for casting (flash_attn requires float16/bfloat16)
model_dtype = next(model.parameters()).dtype

with torch.no_grad():
    for batch_idx, batch in enumerate(data_loader):
        if batch_idx >= 20:  # Limit for demo
            break
        
        batch_data = model.prepare_data(batch)
        gene_ids = batch_data["gene_ids"].to(device)
        values = batch_data["values"].to(device).to(model_dtype)
        target_values = batch_data["target_values"].to(device)
        
        # Get mask: positions where values != target_values (i.e., masked positions)
        mask = (values.float() != target_values.float()) & (target_values != pad_value)
        
        # Forward pass -- get MLM predictions
        src_key_padding_mask = gene_ids.eq(vocab[vocab.pad_token]).to(device)
        output_dict = model(
            gene_ids, values,
            src_key_padding_mask=src_key_padding_mask,
        )
        predictions = output_dict["mlm_output"].float()
        
        # Collect masked position predictions vs actuals
        if mask.any():
            predicted = predictions[mask].cpu().numpy()
            actual = target_values.float()[mask].cpu().numpy()
            all_predicted.extend(predicted)
            all_actual.extend(actual)
        
        if (batch_idx + 1) % 5 == 0:
            print(f"Processed {batch_idx + 1} batches...")

all_predicted = np.array(all_predicted)
all_actual = np.array(all_actual)
print(f"\nTotal masked CpG values evaluated: {len(all_predicted):,}")

## 4. Evaluate Imputation Quality

In [ ]:
from scipy.stats import pearsonr
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Filter valid range [0, 1] for beta values
valid_mask = (all_actual >= 0) & (all_actual <= 1) & (all_predicted >= 0) & (all_predicted <= 1)
pred = all_predicted[valid_mask]
actual = all_actual[valid_mask]

mae = mean_absolute_error(actual, pred)
rmse = np.sqrt(mean_squared_error(actual, pred))
r2 = r2_score(actual, pred)
pearson_r, pearson_p = pearsonr(actual, pred)

print(f"Imputation Results ({len(pred):,} CpG values):")
print(f"  MAE:       {mae:.4f}")
print(f"  RMSE:      {rmse:.4f}")
print(f"  R\u00b2:        {r2:.4f}")
print(f"  Pearson r: {pearson_r:.4f} (p={pearson_p:.2e})")

## 5. Visualize Results

In [ ]:
import matplotlib.pyplot as plt
from aquarel import load_theme

theme = (
    load_theme("scientific")
    .set_grid(draw=False)
    .set_font(size=15)
    .set_ticks(direction="out")
    .set_axis_labels(pad=10)
)
theme.apply()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: Scatter plot with outline
ax = axes[0]
ax.scatter(actual, pred, s=4, c="black", alpha=0.3, zorder=1)
ax.scatter(actual, pred, s=2, alpha=0.2, c="steelblue", zorder=2)
ax.plot([0, 1], [0, 1], "r--", alpha=0.8, label="y = x")
ax.set_xlabel("Actual Beta Value")
ax.set_ylabel("Predicted Beta Value")
ax.set_title(f"Imputation (r={pearson_r:.3f}, MAE={mae:.4f})")
ax.legend(frameon=False)

# Panel 2: Residual distribution
residuals = pred - actual
ax = axes[1]
ax.hist(residuals, bins=100, density=True, alpha=0.7, color="steelblue", edgecolor="black", linewidth=0.3)
ax.axvline(0, color="red", linestyle="--")
ax.set_xlabel("Prediction Error (Predicted - Actual)")
ax.set_ylabel("Density")
ax.set_title(f"Residual Distribution (mean={residuals.mean():.4f})")

# Panel 3: MAE by methylation level
ax = axes[2]
bins = np.linspace(0, 1, 11)
bin_indices = np.digitize(actual, bins) - 1
bin_maes = []
bin_centers = []
for i in range(len(bins) - 1):
    mask = bin_indices == i
    if mask.sum() > 0:
        bin_maes.append(np.mean(np.abs(residuals[mask])))
        bin_centers.append((bins[i] + bins[i+1]) / 2)
ax.bar(bin_centers, bin_maes, width=0.08, alpha=0.7, color="steelblue", edgecolor="black")
ax.set_xlabel("Methylation Level (Beta)")
ax.set_ylabel("MAE")
ax.set_title("MAE by Methylation Level")

theme.apply_transforms()

plt.savefig("imputation_results.pdf", bbox_inches="tight")
plt.savefig("imputation_results.png", dpi=600, bbox_inches="tight")
plt.show()
print("Figure saved to imputation_results.pdf and imputation_results.png")

## Next Steps

- **[Quickstart](../quickstart/)** -- Get started with MethylGPT in minutes
- **[Extract Embeddings](../get_embeddings/)** -- Generate sample-level embeddings for downstream tasks
- **[Age Prediction](../finetuning_age_prediction/)** -- Fine-tune MethylGPT for biological age prediction
- **[Disease Prediction](../disease_prediction/)** -- Fine-tune for disease risk prediction
- **[CpG Selection](../cpg_selection/)** -- Analyze CpG importance and selection